# Agentic Security Lab Round 2 Training Notebook

This notebook keeps the existing `round2_training_colab.ipynb` entrypoint and turns it into a runnable pipeline for:

1. installing OpenEnv and training dependencies,
2. starting the local environment server,
3. collecting live-environment trajectories,
4. training a LoRA policy with TRL,
5. training the world model,
6. exporting metrics and plots.

In [ ]:
!git clone https://github.com/<your-org>/agentic_security_lab.git
%cd agentic_security_lab
!pip install -q -U pip
!pip install -q -e .[train]
!pip install -q openenv-core unsloth

In [ ]:
import os
import subprocess
import time

server = subprocess.Popen([
    'python', '-m', 'uvicorn', 'server.app:app', '--host', '0.0.0.0', '--port', '8000'
])
time.sleep(5)
os.environ['ENV_BASE_URL'] = 'http://127.0.0.1:8000'
print('Server started on', os.environ['ENV_BASE_URL'])

In [ ]:
# Optional baseline run. Set HF_TOKEN / API_BASE_URL / MODEL_NAME first if you want LLM evaluation.
!python inference.py

In [ ]:
!python training/train_grpo.py \
  --env-base-url http://127.0.0.1:8000 \
  --task medium \
  --episodes 12 \
  --model-name Qwen/Qwen2.5-0.5B-Instruct \
  --output-dir artifacts/checkpoints/policy

In [ ]:
!python world_model/train_world_model.py --transitions artifacts/transitions.jsonl --output artifacts/world_model.json
!python training/evaluate.py
!python training/plot_metrics.py
!ls -lah artifacts

In [ ]:
server.terminate()
server.wait()
print('Server stopped.')